In [7]:
import os
import time
import random
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import SessionNotCreatedException, WebDriverException
import numpy as np

# 설정 값
cartoon = "special"
front_url = "https://fanfox.net/manga/sundome_milky_way/v03/c059.6"
chap_str = "059.6"

print('프로그램 시작')

options = Options()
options.add_argument("--headless")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
options.add_argument("--window-size=1920,1080")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

os.makedirs(cartoon, exist_ok=True)

# 챕터 루프 (현재는 1개 챕터만 설정됨)
for chapter in range(0, 1):    
    os.makedirs(f"{cartoon}/{chap_str}", exist_ok=True)  

    # 페이지 루프
    for page in range(1, 500):
        retry_count = 0
        chapter_finished = False  # 해당 챕터가 끝났는지 확인하는 플래그 변수
        
        while retry_count < 5:        
            base_url = front_url + "/1.html#ipg" + str(page)
            url = base_url
            print(f"\n===== 접속: {url} (시도 {retry_count+1}/5)")

            driver = None
            try:
                driver = webdriver.Chrome(options=options)
                driver.get(url)
                time.sleep(4)

                # 성인 인증/주의 문구 클릭
                try:
                    continue_btn = driver.find_element(By.PARTIAL_LINK_TEXT, "click here to continue")
                    if continue_btn.is_displayed():
                        print("성인 인증 버튼 감지: 클릭 시도")
                        continue_btn.click()
                        time.sleep(2)
                except:
                    pass

                driver.execute_script("window.scrollTo(0, 500);")
                time.sleep(1)

                imgs = driver.find_elements(By.CLASS_NAME, "reader-main-img")
                print(f"페이지 {page} - 찾은 이미지 개수: {len(imgs)}")

                # [질문하신 부분] 이미지가 없고 특정 페이지 이상이면 챕터 전체 종료
                if (len(imgs) == 0) and (page > 3):
                    print("이미지 없음 → 챕터 종료 프로세스 시작")
                    chapter_finished = True  # 플래그를 True로 설정
                    driver.quit()
                    break # while 루프 탈출

                page_str = "img_" + str(page).zfill(3)
                gif_detected = False

                for img in imgs:
                    img_url = img.get_attribute("src")
                    if not img_url or "loading.gif" in img_url:
                        time.sleep(3)
                        img_url = img.get_attribute("src")

                    if img_url.startswith("//"):
                        img_url = "https:" + img_url

                    # 이미지 다운로드 (세션 유지)
                    cookies = driver.get_cookies()
                    session = requests.Session()
                    for cookie in cookies:
                        session.cookies.set(cookie['name'], cookie['value'])

                    headers = {"User-Agent": "Mozilla/5.0", "Referer": url}
                    img_res = session.get(img_url, headers=headers)
                    img_data = img_res.content
                    filename = img_url.split("/")[-1].split("?")[0]
                    _, ext = os.path.splitext(filename)

                    # 로딩 이미지가 GIF로 잡히는 경우 재시도
                    if ext.lower() == '.gif' and "loading" not in filename:
                        print("GIF(로딩중) 감지 - 재시도합니다.")
                        retry_count += 1
                        gif_detected = True
                        driver.quit()
                        break 

                    with open(f"{cartoon}/{chap_str}/{page_str}{ext}", "wb") as f:
                        f.write(img_data)
                    print(f"다운로드 완료: {page_str}{ext}")

                if gif_detected:
                    continue 

                driver.quit()
                break # 페이지 작업 성공 시 while 루프 탈출

            except (SessionNotCreatedException, WebDriverException) as e:
                print(f"브라우저 세션 에러 발생: {e}")
                retry_count += 1
                if driver:
                    try: driver.quit()
                    except: pass
                time.sleep(5)

            except Exception as e:
                print(f"일반 에러 발생: {e}")
                retry_count += 1
                if driver:
                    try: driver.quit()
                    except: pass
                time.sleep(3)

        # [수정 포인트] while 루프를 빠져나온 후, 챕터 종료 신호가 있으면 for 루프도 break
        if chapter_finished:
            break 

    print(f"--- 챕터 {chap_str} 작업 완료 ---")

print("모든 작업이 완료되었습니다.")

프로그램 시작

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg1 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 1 - 찾은 이미지 개수: 1
다운로드 완료: img_001.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg2 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 2 - 찾은 이미지 개수: 1
다운로드 완료: img_002.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg3 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 3 - 찾은 이미지 개수: 1
다운로드 완료: img_003.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg4 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 4 - 찾은 이미지 개수: 1
다운로드 완료: img_004.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg5 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 5 - 찾은 이미지 개수: 1
다운로드 완료: img_005.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg6 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 6 - 찾은 이미지 개수: 1
다운로드 완료: img_006.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg7 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 7

성인 인증 버튼 감지: 클릭 시도
페이지 54 - 찾은 이미지 개수: 1
다운로드 완료: img_054.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg55 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 55 - 찾은 이미지 개수: 1
다운로드 완료: img_055.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg56 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 56 - 찾은 이미지 개수: 1
다운로드 완료: img_056.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg57 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 57 - 찾은 이미지 개수: 1
다운로드 완료: img_057.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg58 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 58 - 찾은 이미지 개수: 1
다운로드 완료: img_058.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg59 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 59 - 찾은 이미지 개수: 1
다운로드 완료: img_059.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg60 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 60 - 찾은 이미지 개수: 1
다운로드 완료: img_060.jpg

===== 접속: https://fanfox.net/manga/sundome_


===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg108 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 108 - 찾은 이미지 개수: 1
다운로드 완료: img_108.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg109 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 109 - 찾은 이미지 개수: 1
다운로드 완료: img_109.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg110 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 110 - 찾은 이미지 개수: 1
다운로드 완료: img_110.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg111 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 111 - 찾은 이미지 개수: 1
다운로드 완료: img_111.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg112 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 112 - 찾은 이미지 개수: 1
다운로드 완료: img_112.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg113 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 113 - 찾은 이미지 개수: 1
다운로드 완료: img_113.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg114 (시도 1/5)
성인 인증 

성인 인증 버튼 감지: 클릭 시도
페이지 162 - 찾은 이미지 개수: 1
다운로드 완료: img_162.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg163 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 163 - 찾은 이미지 개수: 1
다운로드 완료: img_163.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg164 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 164 - 찾은 이미지 개수: 1
다운로드 완료: img_164.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg165 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 165 - 찾은 이미지 개수: 1
다운로드 완료: img_165.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg166 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 166 - 찾은 이미지 개수: 1
다운로드 완료: img_166.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg167 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 167 - 찾은 이미지 개수: 1
다운로드 완료: img_167.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg168 (시도 1/5)
성인 인증 버튼 감지: 클릭 시도
페이지 168 - 찾은 이미지 개수: 1
다운로드 완료: img_168.jpg

===== 접속: https://fanfox.net/m

성인 인증 버튼 감지: 클릭 시도
페이지 215 - 찾은 이미지 개수: 1
다운로드 완료: img_215.jpg

===== 접속: https://fanfox.net/manga/sundome_milky_way/v03/c059.6/1.html#ipg216 (시도 1/5)
페이지 216 - 찾은 이미지 개수: 0
이미지 없음 → 챕터 종료 프로세스 시작
--- 챕터 059.6 작업 완료 ---
모든 작업이 완료되었습니다.
